# 06 - Entrenar modelo principal

Transfer learning con una red preentrenada (ImageNet) + cabeza clasificadora binaria. Se espera superar al baseline.

## 1. Setup y datos

In [ ]:
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

REPO = Path.cwd()
dataset_root = REPO / "data" / "raw" / "chest_xray"
if not dataset_root.exists():
    dataset_root = REPO / "data" / "raw"
models_dir = REPO / "models"
models_dir.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.15
EPOCHS_FREEZE = 5
EPOCHS_FINETUNE = 15

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode="nearest",
    validation_split=VAL_SPLIT,
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=VAL_SPLIT)

# Train y val desde la carpeta "train" vía validation_split; test desde "test"
train_ds = train_datagen.flow_from_directory(
    dataset_root / "train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED,
    subset="training",
)
val_ds = val_datagen.flow_from_directory(
    dataset_root / "train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
    subset="validation",
)
test_ds = val_datagen.flow_from_directory(
    dataset_root / "test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)
n_train = train_ds.samples
n_pos = int((train_ds.classes == 1).sum())
n_neg = n_train - n_pos
class_weight = {0: n_train / (2 * n_neg), 1: n_train / (2 * n_pos)}
print(f"Train: {n_train} (NORMAL: {n_neg}, PNEUMONIA: {n_pos})")
print("Class weights:", class_weight)

## 2. Backbones y bucle de entrenamiento

Entrenamos varios modelos clásicos de transfer learning (EfficientNet, VGG, ResNet, DenseNet) con la misma lógica que el baseline: fase 1 con backbone congelado, fase 2 con fine-tuning, callbacks (EarlyStopping + ReduceLROnPlateau), class_weight y evaluación en test.

In [ ]:
# Backbones clásicos (nombre, clase de keras.applications)
BACKBONES = [
    ("EfficientNetB0", keras.applications.EfficientNetB0),
    ("VGG16", keras.applications.VGG16),
    ("ResNet50", keras.applications.ResNet50),
    ("DenseNet121", keras.applications.DenseNet121),
    ("MobileNetV2", keras.applications.MobileNetV2),
]

def build_model_with_backbone(BackboneClass, input_shape=(224, 224, 3)):
    base = BackboneClass(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )
    base.trainable = False
    model = keras.Sequential([
        base,
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])
    return model, base

results = []

## 3. Bucle: fase 1 (cabeza) + fase 2 (fine-tuning) por backbone

In [ ]:
for name, BackboneClass in BACKBONES:
    print(f"\n{'='*60}\nEntrenando: {name}\n{'='*60}")
    model, base = build_model_with_backbone(BackboneClass, (*IMG_SIZE, 3))
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )

    # Fase 1: solo cabeza
    callbacks_1 = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True,
        ),
    ]
    history1 = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_FREEZE,
        class_weight=class_weight, callbacks=callbacks_1,
    )

    # Fase 2: fine-tuning (últimas capas del backbone)
    base.trainable = True
    n_freeze = min(30, len(base.layers) - 1)
    for layer in base.layers[:-n_freeze]:
        layer.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )
    callbacks_2 = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7,
        ),
    ]
    history2 = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
        class_weight=class_weight, callbacks=callbacks_2,
    )

    path_save = models_dir / f"main_{name.lower()}.keras"
    model.save(path_save)
    print(f"Guardado: {path_save}")
    loss, acc, auc = model.evaluate(test_ds)
    results.append({
        "name": name,
        "test_loss": float(loss),
        "test_accuracy": float(acc),
        "test_auc": float(auc),
        "history1": history1.history,
        "history2": history2.history,
    })
    print(f"Test — loss: {loss:.4f}, accuracy: {acc:.4f}, AUC: {auc:.4f}")

## 4. Resumen de resultados en test

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {"modelo": r["name"], "test_loss": r["test_loss"], "test_accuracy": r["test_accuracy"], "test_auc": r["test_auc"]}
    for r in results
])
summary = summary.sort_values("test_auc", ascending=False)
display(summary)

## 5. Curvas de entrenamiento (loss, accuracy, AUC)

In [ ]:
import matplotlib.pyplot as plt

for r in results:
    name = r["name"]
    h1, h2 = r["history1"], r["history2"]
    epochs1 = range(1, len(h1["loss"]) + 1)
    epochs2 = range(len(epochs1) + 1, len(epochs1) + len(h2["loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    # Loss
    axes[0].plot(epochs1, h1["loss"], "b-", label="Train (fase 1)")
    axes[0].plot(epochs1, h1["val_loss"], "b--", label="Val (fase 1)")
    axes[0].plot(epochs2, h2["loss"], "g-", label="Train (fase 2)")
    axes[0].plot(epochs2, h2["val_loss"], "g--", label="Val (fase 2)")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title(f"{name} — Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    # Accuracy
    axes[1].plot(epochs1, h1["accuracy"], "b-"); axes[1].plot(epochs1, h1["val_accuracy"], "b--")
    axes[1].plot(epochs2, h2["accuracy"], "g-"); axes[1].plot(epochs2, h2["val_accuracy"], "g--")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].set_title(f"{name} — Accuracy"); axes[1].legend(["Train (f1)", "Val (f1)", "Train (f2)", "Val (f2)"]); axes[1].grid(True, alpha=0.3)
    # AUC
    axes[2].plot(epochs1, h1["auc"], "b-"); axes[2].plot(epochs1, h1["val_auc"], "b--")
    axes[2].plot(epochs2, h2["auc"], "g-"); axes[2].plot(epochs2, h2["val_auc"], "g--")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("AUC"); axes[2].set_title(f"{name} — AUC"); axes[2].legend(["Train (f1)", "Val (f1)", "Train (f2)", "Val (f2)"]); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()